# 01 — Model definition

### How to read this notebook
Run cells top to bottom. Each Markdown cell states the question and expected result **before** the code. The printed and plotted numbers are measurements, not claims made in advance. The shared functions live in `src/experiment.py`. Open Jupyter from the project root so imports work.

## Architecture and shapes

The input is $(B,3)$. `fc1.weight` is $(32,3)$: column 0 corresponds to the simple feature, columns 1–2 to the complex pair. The hidden preactivation is $Z_1=XW_1^\top+b_1$ with shape $(B,32)$. ReLU produces $H=\max(0,Z_1)$. `fc2` produces logits $(B,1)$; `BCEWithLogitsLoss` applies the sigmoid internally.

The weight columns are **slices of one matrix**, not separate neural networks.

In [1]:
from pathlib import Path
import os
import sys

# Resolve this experiment before imports and relative result paths.
ROOT = Path.cwd()
if not (ROOT / 'src' / 'experiment.py').is_file():
    ROOT = ROOT / 'experiment_1'
assert (ROOT / 'src' / 'experiment.py').is_file(), 'Start Jupyter in SimpBias or experiment_1.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.experiment import *
import torch
import numpy as np
import matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)

PyTorch: 2.14.1+cpu


In [2]:
model=make_model(42)
x,y=make_data(8,42)
z1=model.fc1(x)
h=model.relu(z1)
logits=model(x)
print('X:',tuple(x.shape),'W1:',tuple(model.fc1.weight.shape),'Z1:',tuple(z1.shape),'H:',tuple(h.shape),'logits:',tuple(logits.shape))
print('Simple weight column:',tuple(model.fc1.weight[:,0:1].shape))
print('Complex weight columns:',tuple(model.fc1.weight[:,1:3].shape))

X: (8, 3) W1: (32, 3) Z1: (8, 32) H: (8, 32) logits: (8, 1)
Simple weight column: (32, 1)
Complex weight columns: (32, 2)


## ReLU gates and gradients

For a hidden unit $j$, $z_j=w_{js}x_s+\alpha w_{jc1}x_{c1}+\alpha w_{jc2}x_{c2}+b_j$. The gate is $\mathbf1[z_j>0]$. The gradient to a complex weight includes the scaled input $\alpha x_c$ and the gate. Inspect these quantities below.

In [3]:
loss=torch.nn.BCEWithLogitsLoss()(model(x),y)
loss.backward()
print('Active ReLU fraction:',(z1>0).float().mean().item())
print('First-layer gradient shape:',tuple(model.fc1.weight.grad.shape))
print('Mean abs gradient, simple:',model.fc1.weight.grad[:,0].abs().mean().item())
print('Mean abs gradient, complex:',model.fc1.weight.grad[:,1:3].abs().mean().item())

Active ReLU fraction: 0.390625
First-layer gradient shape: (32, 3)
Mean abs gradient, simple: 0.019621439278125763
Mean abs gradient, complex: 0.01373680867254734
